In [1]:
#!/usr/bin/env python3
"""
fetch_resources_spark.py

PySpark script pour Synapse Spark Pool qui télécharge customers + items + invoices + contracts + transactions (+ revenue_entries)
depuis SaaSOptics et écrit les résultats en Parquet.

Remarque importante pour Data Flow :
- Pour éviter les erreurs lors de la lecture par un Data Flow (problèmes liés aux colonnes de type tableau / struct imbriquées),
  le fichier Parquet invoices écrit est désormais "plat" : la colonne `line_items` n'est PAS incluse dans le parquet invoices.
- Les lignes d'invoices (line_items) sont écrites séparément dans invoices_line_items (une table/chemin Parquet sans struct imbriqué).
- J'ai aussi aplati le bloc `auditentry` des invoices en colonnes simples (audit_created, audit_modified, ...), ce qui facilite la lecture
  depuis un Data Flow qui a parfois des limitations sur les structs imbriquées.

Ajout :
- Téléchargement des transactions (endpoint /transactions/)
- Téléchargement des revenue entries pour chaque transaction (endpoint /transactions/{transaction_pk}/revenue_entries/)
- Normalization basique pour les revenue_entries et écriture en parquet sous OUTPUT_BASE/revenue_entries/

Les méthodes de connexion / récupération du secret (Key Vault) n'ont pas été modifiées.
"""
import os
import time
import json
from urllib.parse import urljoin, urlparse
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, to_date, explode_outer

# ========== Configuration ==========
CUSTOMERS_ENDPOINT = os.environ.get(
    "CUSTOMERS_ENDPOINT",
    "https://s12.saasoptics.com/qbdv10_lucid/api/v1.0/customers/"
)
ITEMS_ENDPOINT = os.environ.get(
    "ITEMS_ENDPOINT",
    "https://s12.saasoptics.com/qbdv10_lucid/api/v1.0/items/"
)
INVOICES_ENDPOINT = os.environ.get(
    "INVOICES_ENDPOINT",
    "https://s12.saasoptics.com/qbdv10_lucid/api/v1.0/invoices/"
)
CONTRACTS_ENDPOINT = os.environ.get(
    "CONTRACTS_ENDPOINT",
    "https://s12.saasoptics.com/qbdv10_lucid/api/v1.0/contracts/"
)
TRANSACTIONS_ENDPOINT = os.environ.get(
    "TRANSACTIONS_ENDPOINT",
    "https://s12.saasoptics.com/qbdv10_lucid/api/v1.0/transactions/"
)
# revenue entries endpoint will be built from TRANSACTIONS_ENDPOINT + "{id}/revenue_entries/"

# By default you can set these env vars in Synapse Spark job configuration
KEY_VAULT_LINKED_SERVICE = os.environ.get("KEY_VAULT_LINKED_SERVICE", "DCIST_DatalakeSynapse")
KEY_VAULT_SECRET_NAME = os.environ.get("KEY_VAULT_SECRET_NAME", "MaxioToken")

# fallback env var if Key Vault not used or unavailable
API_KEY_ENV = os.environ.get("MAXIO_API_KEY")

AUTH_TYPE = os.environ.get("MAXIO_AUTH_TYPE", "bearer")
PAGE_SIZE = int(os.environ.get("PAGE_SIZE", "100"))
# default OUTPUT_BASE should be overridden in Synapse to an ADLS path; fallback to /tmp for local testing
datalakeStorageAccount = json.loads(mssparkutils.credentials.getPropertiesAll("dcist_synapseenvironement_workspace"))["Endpoint"]
datalakeStorageAccount = datalakeStorageAccount[8:]
OUTPUT_BASE =  "abfss://dcist-bronze-container@"+datalakeStorageAccount+"Bronze_Maxio/"
NUM_PARTITIONS = int(os.environ.get("NUM_PARTITIONS", "200"))
REQUESTS_PER_SECOND = float(os.environ.get("MAXIO_RPS", "5"))
# ===================================

def get_api_key_from_synapse_keyvault(linked_service_name: str, secret_name: str) -> str:
    """
    Try to get secret from a Synapse Linked Service Key Vault using mssparkutils.
    This function works only inside Synapse notebooks / Spark pools where mssparkutils is available.
    """
  
    try:
        # mssparkutils.credentials.getPropertiesAll expects the linked service name or its identifier
        secret_value =  mssparkutils.credentials.getSecretWithLS(linked_service_name, secret_name)
        if secret_value is None or secret_value == "":
            raise RuntimeError(f"Secret '{secret_name}' récupéré depuis linked service '{linked_service_name}' est vide.")
        return secret_value
    except Exception as e:
        raise RuntimeError(f"Impossible de récupérer le secret '{secret_name}' depuis linked service '{linked_service_name}': {e}") from e

def get_api_key():
    """
    Retourne la clé API en essayant, dans l'ordre:
    1) Key Vault linked service (KEY_VAULT_LINKED_SERVICE + KEY_VAULT_SECRET_NAME) via mssparkutils
    2) variable d'environnement MAXIO_API_KEY
    Si aucune n'est trouvée, lève une exception.
    """
    # 1) try Key Vault via linked service (only in Synapse)
    if KEY_VAULT_LINKED_SERVICE and KEY_VAULT_SECRET_NAME:
        try:
            api_key = get_api_key_from_synapse_keyvault(KEY_VAULT_LINKED_SERVICE, KEY_VAULT_SECRET_NAME)
            print("Clé API récupérée depuis Key Vault linked service.")
            return api_key
        except Exception as e:
            # log and fall back
            print(f"[WARN] Impossible de récupérer la clé depuis Key Vault linked service: {e}. Tentative fallback sur env var.")

    # 2) fallback env var
    if API_KEY_ENV:
        print("Clé API récupérée depuis la variable d'environnement MAXIO_API_KEY.")
        return API_KEY_ENV

    raise RuntimeError("Clé API introuvable. Définir KEY_VAULT_LINKED_SERVICE + KEY_VAULT_SECRET_NAME (Synapse) ou MAXIO_API_KEY (env).")


def build_headers():
    return {
        "Authorization": f"Token {get_api_key()}",
        "Accept": "application/json",
        "Content-Type": "application/json",
        "User-Agent": "spark-resource-client/1.0"
    }

def list_resources_via_results(session, url, headers, page_size=100):
    """Paginate via 'next' and collect all objects in 'results' (or list fallback)."""
    items = []
    next_url = url
    params = {"page_size": page_size}
    while next_url:
        resp = session.get(next_url, headers=headers, params=params, timeout=60)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", "5"))
            time.sleep(wait)
            continue
        resp.raise_for_status()
        j = resp.json()
        if isinstance(j, dict) and "results" in j:
            page_items = j.get("results") or []
            items.extend(page_items)
            next_url = j.get("next")
            if next_url:
                parsed = urlparse(next_url)
                if not parsed.scheme:
                    next_url = urljoin(url, next_url)
            params = {}
            continue
        else:
            if isinstance(j, list):
                items.extend(j)
            break
    return items

def fetch_resource_details_partition(iterable_ids_or_objs, base_url, b_headers, rps_limit=REQUESTS_PER_SECOND):
    """
    If element is dict -> yield directly.
    Else assume it's an id -> GET base_url/{id}.
    """
    import requests
    import time
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry

    headers = b_headers.value
    session = requests.Session()
    retries = Retry(total=5, backoff_factor=0.5, status_forcelist=[429,500,502,503,504], raise_on_status=False)
    session.mount("https://", HTTPAdapter(max_retries=retries))
    session.mount("http://", HTTPAdapter(max_retries=retries))

    min_interval = 1.0 / max(rps_limit, 1.0)
    last_call = 0.0
    base_url = base_url.rstrip('/')

    for element in iterable_ids_or_objs:
        if isinstance(element, dict):
            yield element
            continue

        iid = element
        url = f"{base_url}/{iid}"
        backoff = 1.0
        while True:
            now = time.time()
            sleep_for = min_interval - (now - last_call)
            if sleep_for > 0:
                time.sleep(sleep_for)
            try:
                resp = session.get(url, headers=headers, timeout=60)
            except Exception:
                time.sleep(backoff)
                backoff = min(backoff * 2, 60)
                continue
            last_call = time.time()
            if resp.status_code == 200:
                try:
                    yield resp.json()
                except Exception:
                    yield {"id": iid, "_error": "bad_json", "raw": resp.text}
                break
            elif resp.status_code == 404:
                yield {"id": iid, "_error": "not_found", "status_code": 404}
                break
            elif resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", backoff))
                time.sleep(wait)
                backoff = min(backoff * 2, 60)
                continue
            elif 500 <= resp.status_code < 600:
                time.sleep(backoff)
                backoff = min(backoff * 2, 60)
                continue
            else:
                try:
                    yield {"id": iid, "_error": "http", "status_code": resp.status_code, "body": resp.text}
                except Exception:
                    yield {"id": iid, "_error": "http", "status_code": resp.status_code}
                break

def _sanitize_complex_columns(df):
    """
    Convert any complex column (StructType, ArrayType, MapType) into JSON string using to_json.
    This prevents Data Flow / downstream readers from failing on non-primitive Parquet columns.
    """
    from pyspark.sql.types import ArrayType, StructType, MapType
    from pyspark.sql.functions import to_json

    complex_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, (ArrayType, StructType, MapType))]
    if not complex_cols:
        return df
    # Convert each complex column to JSON string (overwriting the same column name)
    for c in complex_cols:
        # to_json handles struct/array/map types
        df = df.withColumn(c, to_json(col(c)))
    return df

def process_resource(spark, sc, name, endpoint_url, headers, normalize_func=None):
    """Generic resource processing: list, parallelize, convert to DF, normalize, write parquet."""
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry

    s = requests.Session()
    retries = Retry(total=5, backoff_factor=0.5, status_forcelist=[429,500,502,503,504], raise_on_status=False)
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.mount("http://", HTTPAdapter(max_retries=retries))

    print(f"Listing resource {name} from {endpoint_url} ...")
    items = list_resources_via_results(s, endpoint_url, headers, page_size=PAGE_SIZE)
    print(f"Retrieved {len(items)} items for {name}")

    if len(items) == 0:
        print(f"Aucun élément pour {name}. Skip.")
        return

    b_headers = sc.broadcast(headers)
    first = items[0]
    use_direct_objects = isinstance(first, dict)

    if use_direct_objects:
        rdd = sc.parallelize(items, NUM_PARTITIONS)
        details_rdd = rdd.map(lambda d: d)
    else:
        rdd = sc.parallelize(items, NUM_PARTITIONS)
        details_rdd = rdd.mapPartitions(lambda it: fetch_resource_details_partition(it, endpoint_url, b_headers, REQUESTS_PER_SECOND))

    # raw_df contains potential nested 'line_items' for invoices -> we keep this to write invoice lines separately
    json_rdd = details_rdd.map(lambda d: json.dumps(d, ensure_ascii=False))
    raw_df = spark.read.json(json_rdd)

    print(f"Inferred schema for {name}:")
    raw_df.printSchema()
    print(f"Count (approx) for {name}: {raw_df.count()}")

    # If this is invoices, write invoice lines from raw_df BEFORE dropping the nested column
    if name == "invoices":
        out_line_items_path = os.path.join(OUTPUT_BASE.rstrip('/'), "invoices_line_items")
        try:
            write_invoice_lines_from_df(spark, raw_df, out_line_items_path)
        except Exception as e:
            print(f"[WARN] Échec écriture invoice lines from raw_df: {e}")

    # If this is transactions, write revenue_entries for each transaction BEFORE normalization (so we use raw ids)
    if name == "transactions":
        out_revenue_entries_path = os.path.join(OUTPUT_BASE.rstrip('/'), "revenue_entries")
        try:
            write_revenue_entries_for_transactions(spark, sc, raw_df, out_revenue_entries_path, endpoint_url, headers)
        except Exception as e:
            print(f"[WARN] Échec écriture revenue entries for transactions: {e}")

    # then normalize (drop nested `line_items` from normalized invoices)
    if normalize_func:
        df_norm = normalize_func(raw_df)
    else:
        df_norm = raw_df

    # ensure line_items column is not present in the invoices parquet to avoid Data Flow issues
    if name == "invoices" and "line_items" in df_norm.columns:
        df_norm = df_norm.drop("line_items")

    # SANITIZE: convert any remaining complex columns to JSON strings (handles 'auditentry' or other structs/arrays)
    df_norm = _sanitize_complex_columns(df_norm)

    out_path = os.path.join(OUTPUT_BASE.rstrip('/'), name)
    print(f"Ecriture du résultat ({name}) vers {out_path} ...")
    df_norm.write.mode("overwrite").parquet(out_path)
    print(f"Write complete for {name}")


def normalize_items_df(df):
    """Normalization/casts for items resource based on provided example."""
    return df.select(
        col("id"),
        col("product_family"),
        col("name"),
        col("code"),
        col("source_system"),
        col("description"),
        col("recurring").cast("boolean").alias("recurring"),
        col("billing_description"),
        col("default_duration"),
        col("normalized_term"),
        col("is_active").cast("boolean").alias("is_active"),
        col("enable_create_transactions").cast("boolean").alias("enable_create_transactions"),
        col("create_revenue").cast("boolean").alias("create_revenue"),
        col("use_chargify_start_and_end_dates").cast("boolean").alias("use_chargify_start_and_end_dates"),
        col("needs_so_profile").cast("boolean").alias("needs_so_profile"),
        col("qb_id"),
        col("sf_do_not_sync").cast("boolean").alias("sf_do_not_sync"),
        col("wizard_enabled").cast("boolean").alias("wizard_enabled"),
        col("gl_name"),
        col("gl_description"),
        col("sync_invoices").cast("boolean").alias("sync_invoices"),
        col("is_sales_tax").cast("boolean").alias("is_sales_tax"),
        col("is_discount").cast("boolean").alias("is_discount"),
        col("is_taxable").cast("boolean").alias("is_taxable"),
        col("income_account").cast("long").alias("income_account"),
        col("qb_account").cast("long").alias("qb_account"),
        to_timestamp(col("modified")).alias("modified"),
        col("modified_by_name")
    )

def normalize_invoices_df(df):
    """Normalization/casts for invoices resource (based on provided invoice example).
    NOTE: do not include the 'line_items' column here to keep invoices parquet flat for Data Flow consumption.
    Also flatten auditentry into simple columns.
    """
    return df.select(
        col("id"),
        col("number"),
        to_date(col("date"), "yyyy-MM-dd").alias("date"),
        to_date(col("due_date"), "yyyy-MM-dd").alias("due_date"),
        to_date(col("ship_date"), "yyyy-MM-dd").alias("ship_date"),
        col("po_number"),
        col("memo"),
        col("other"),
        col("to_be_printed").cast("boolean").alias("to_be_printed"),
        col("to_be_emailed").cast("boolean").alias("to_be_emailed"),
        col("billing_addr1"),
        col("billing_addr2"),
        col("billing_addr3"),
        col("billing_city"),
        col("billing_state"),
        col("billing_zip_code"),
        col("billing_country"),
        col("shipping_addr1"),
        col("shipping_addr2"),
        col("shipping_addr3"),
        col("shipping_city"),
        col("shipping_state"),
        col("shipping_zip_code"),
        col("shipping_country"),
        to_timestamp(col("exported_date")).alias("exported_date"),
        col("do_not_sync").cast("boolean").alias("do_not_sync"),
        col("ignore_date_when_syncing").cast("boolean").alias("ignore_date_when_syncing"),
        col("type"),
        col("chargify_id"),
        col("qb_number"),
        col("qb_txn_id"),
        col("deleted_in_qb").cast("boolean").alias("deleted_in_qb"),
        to_timestamp(col("sync_date")).alias("sync_date"),
        col("foreign_exchange_rate").cast("double").alias("foreign_exchange_rate"),
        col("subtotal").cast("double").alias("subtotal"),
        col("is_paid").cast("boolean").alias("is_paid"),
        col("sales_tax").cast("double").alias("sales_tax"),
        col("sales_tax_percentage").cast("double").alias("sales_tax_percentage"),
        col("applied_amount").cast("double").alias("applied_amount"),
        col("balance").cast("double").alias("balance"),
        col("email_from_so").cast("boolean").alias("email_from_so"),
        col("enable_cc_payment"),
        col("enable_ach_payment"),
        col("sf_id"),
        col("stripe_id"),
        col("contract").cast("long").alias("contract"),
        col("qb_payment_terms").cast("long").alias("qb_payment_terms"),
        col("qb_class").cast("long").alias("qb_class"),
        col("qb_template").cast("long").alias("qb_template"),
        col("qb_customer_message"),
        col("qb_sales_rep").cast("long").alias("qb_sales_rep"),
        col("qb_currency"),
        col("qb_ar_account").cast("long").alias("qb_ar_account"),
        col("refund_of"),
        col("external_id"),
        col("committed").cast("boolean").alias("committed"),
        to_timestamp(col("committed_timestamp")).alias("committed_timestamp"),
        # flatten auditentry struct into simple columns
        col("auditentry.created").alias("audit_created"),
        col("auditentry.created_by").alias("audit_created_by"),
        col("auditentry.created_by_name").alias("audit_created_by_name"),
        col("auditentry.modified").alias("audit_modified"),
        col("auditentry.modified_by").alias("audit_modified_by"),
        col("auditentry.modified_by_name").alias("audit_modified_by_name"),
        # do not include 'line_items' here (we write them separately)
        col("local_amount").cast("double").alias("local_amount"),
        col("home_amount").cast("double").alias("home_amount"),
        col("tax_lines"),
        to_timestamp(col("last_emailed")).alias("last_emailed"),
        to_timestamp(col("last_clicked")).alias("last_clicked"),
        to_timestamp(col("last_opened")).alias("last_opened"),
        col("einvoicing_url"),
        col("einvoicing_url_no_click_tracking")
    )

def normalize_customers_df(df):
    """
    Normalize customers and flatten billing_profile fields based on sample structure provided.
    Keeps primary customer fields and flattens billing_profile.* with billing_ prefix.
    """
    return df.select(
        col("id"),
        col("name"),
        col("number"),
        col("code"),
        col("domain"),
        col("notes"),
        col("is_active").cast("boolean").alias("is_active"),
        col("qb_id"),
        col("do_not_sync").cast("boolean").alias("do_not_sync"),
        col("unbalanced_revenue_exception").cast("boolean").alias("unbalanced_revenue_exception"),
        to_timestamp(col("modified")).alias("modified"),
        col("chargify_id"),
        col("sf_id"),
        col("sf_owner_id"),
        col("einvoicing_id"),
        col("email"),
        col("cc_email"),
        col("escalation_email"),
        col("default_email_from_so").cast("boolean").alias("default_email_from_so"),
        col("default_enable_cc_payment").cast("boolean").alias("default_enable_cc_payment"),
        col("default_enable_ach_payment").cast("boolean").alias("default_enable_ach_payment"),
        col("autopay_enrollment"),
        col("autopay_status"),
        col("parent"),
        col("default_theme"),
        col("is_paying_customer").cast("boolean").alias("is_paying_customer"),
        col("text_field1"),
        col("text_field2"),
        col("text_field3"),
        col("number_field1"),
        col("number_field2"),
        col("number_field3"),
        col("industry"),
        col("segment"),
        col("market"),
        col("c_account_type"),
        col("c_po_reqd_for_invoicing"),
        col("c_salesforce_id"),
        col("c_subvertical"),
        col("c_vertical"),
        # auditentry kept as nested struct (customers parquet may contain this; sanitizer will convert if needed)
        col("auditentry"),
        # Flatten billing_profile
        col("billing_profile.name").alias("billing_name"),
        col("billing_profile.company_name").alias("billing_company_name"),
        col("billing_profile.salutation").alias("billing_salutation"),
        col("billing_profile.first_name").alias("billing_first_name"),
        col("billing_profile.last_name").alias("billing_last_name"),
        col("billing_profile.addr1").alias("billing_addr1"),
        col("billing_profile.addr2").alias("billing_addr2"),
        col("billing_profile.addr3").alias("billing_addr3"),
        col("billing_profile.city").alias("billing_city"),
        col("billing_profile.state").alias("billing_state"),
        col("billing_profile.zip_code").alias("billing_zip_code"),
        col("billing_profile.country").alias("billing_country"),
        col("billing_profile.shipping_addr1").alias("shipping_addr1"),
        col("billing_profile.shipping_addr2").alias("shipping_addr2"),
        col("billing_profile.shipping_addr3").alias("shipping_addr3"),
        col("billing_profile.shipping_city").alias("shipping_city"),
        col("billing_profile.shipping_state").alias("shipping_state"),
        col("billing_profile.shipping_zip_code").alias("shipping_zip_code"),
        col("billing_profile.shipping_country").alias("shipping_country"),
        col("billing_profile.phone").alias("billing_phone"),
        col("billing_profile.alt_phone").alias("billing_alt_phone"),
        col("billing_profile.fax").alias("billing_fax"),
        col("billing_profile.email").alias("billing_email"),
        col("billing_profile.contact").alias("billing_contact"),
        col("billing_profile.alt_contact").alias("billing_alt_contact"),
        col("billing_profile.sales_rep_fullname").alias("billing_sales_rep_fullname"),
        col("billing_profile.resale_number").alias("billing_resale_number"),
        col("billing_profile.account_number").alias("billing_account_number"),
        col("billing_profile.is_active").cast("boolean").alias("billing_is_active"),
        col("billing_profile.invoice_print_preference").cast("boolean").alias("billing_invoice_print_preference"),
        col("billing_profile.invoice_email_preference").cast("boolean").alias("billing_invoice_email_preference"),
        col("billing_profile.edit_sequence").alias("billing_edit_sequence"),
        col("billing_profile.last_updated_by_qb").cast("boolean").alias("billing_last_updated_by_qb"),
        to_date(col("billing_profile.avatax_address_validation_timestamp")).alias("billing_avatax_address_validation_timestamp"),
        col("billing_profile.payment_terms").cast("long").alias("billing_payment_terms"),
        col("billing_profile.payment_method").alias("billing_payment_method"),
        col("billing_profile.currency").alias("billing_currency"),
        col("billing_profile.sales_tax_code").alias("billing_sales_tax_code"),
        col("billing_profile.item_sales_tax").alias("billing_item_sales_tax"),
        col("billing_profile.sales_rep").alias("billing_sales_rep"),
        col("billing_profile.customer_type").alias("billing_customer_type"),
        col("billing_profile.default_invoice_template").alias("billing_default_invoice_template"),
        col("billing_profile.default_credit_memo_template").alias("billing_default_credit_memo_template"),
        col("billing_profile.default_class").alias("billing_default_class"),
        col("billing_profile.entity_use_code").alias("billing_entity_use_code"),
        col("billing_profile.tax_exemption_reason").alias("billing_tax_exemption_reason")
    )

def normalize_contracts_df(df):
    """
    Normalize contracts and flatten billing_profile fields based on sample structure provided.
    Keeps primary contract fields and flattens billing_profile.* with billing_ prefix.
    """
    return df.select(
        col("id"),
        col("number"),
        col("maxio_id"),
        col("entry_date"),
        col("is_active").cast("boolean").alias("is_active"),
        col("notes"),
        col("qb_job_id"),
        col("is_job").cast("boolean").alias("is_job"),
        col("unbalanced_revenue_exception").cast("boolean").alias("unbalanced_revenue_exception"),
        col("email"),
        col("default_cadence_template"),
        col("sf_id"),
        col("register").cast("long").alias("register"),
        col("register_maxio_id"),
        col("customer").cast("long").alias("customer"),
        col("created"),
        col("created_by"),
        col("created_by_name"),
        to_timestamp(col("modified")).alias("modified"),
        col("modified_by"),
        col("modified_by_name"),
        col("text_field1"),
        col("text_field2"),
        col("number_field1"),
        col("number_field2"),
        col("channel"),
        col("lead_source"),
        to_date(col("lead_date")).alias("lead_date"),
        # Flatten billing_profile
        col("billing_profile.parent_id").alias("billing_parent_id"),
        col("billing_profile.name").alias("billing_name"),
        col("billing_profile.company_name").alias("billing_company_name"),
        col("billing_profile.salutation").alias("billing_salutation"),
        col("billing_profile.first_name").alias("billing_first_name"),
        col("billing_profile.last_name").alias("billing_last_name"),
        col("billing_profile.addr1").alias("billing_addr1"),
        col("billing_profile.addr2").alias("billing_addr2"),
        col("billing_profile.addr3").alias("billing_addr3"),
        col("billing_profile.city").alias("billing_city"),
        col("billing_profile.state").alias("billing_state"),
        col("billing_profile.zip_code").alias("billing_zip_code"),
        col("billing_profile.country").alias("billing_country"),
        col("billing_profile.shipping_addr1").alias("billing_shipping_addr1"),
        col("billing_profile.shipping_addr2").alias("billing_shipping_addr2"),
        col("billing_profile.shipping_addr3").alias("billing_shipping_addr3"),
        col("billing_profile.shipping_city").alias("billing_shipping_city"),
        col("billing_profile.shipping_state").alias("billing_shipping_state"),
        col("billing_profile.shipping_zip_code").alias("billing_shipping_zip_code"),
        col("billing_profile.shipping_country").alias("billing_shipping_country"),
        col("billing_profile.phone").alias("billing_phone"),
        col("billing_profile.alt_phone").alias("billing_alt_phone"),
        col("billing_profile.fax").alias("billing_fax"),
        col("billing_profile.email").alias("billing_email"),
        col("billing_profile.contact").alias("billing_contact"),
        col("billing_profile.alt_contact").alias("billing_alt_contact"),
        col("billing_profile.customer_type_ref_fullname").alias("billing_customer_type_ref_fullname"),
        col("billing_profile.resale_number").alias("billing_resale_number"),
        col("billing_profile.account_number").alias("billing_account_number"),
        col("billing_profile.is_active").cast("boolean").alias("billing_is_active"),
        col("billing_profile.job_status").alias("billing_job_status"),
        to_date(col("billing_profile.job_start_date")).alias("billing_job_start_date"),
        to_date(col("billing_profile.job_projected_end_date")).alias("billing_job_projected_end_date"),
        to_date(col("billing_profile.job_end_date")).alias("billing_job_end_date"),
        col("billing_profile.job_description").alias("billing_job_description"),
        col("billing_profile.invoice_print_preference").cast("boolean").alias("billing_invoice_print_preference"),
        col("billing_profile.invoice_email_preference").cast("boolean").alias("billing_invoice_email_preference"),
        col("billing_profile.edit_sequence").alias("billing_edit_sequence"),
        col("billing_profile.payment_terms").cast("long").alias("billing_payment_terms"),
        col("billing_profile.sales_rep").cast("long").alias("billing_sales_rep"),
        col("billing_profile.job_type").alias("billing_job_type"),
        col("billing_profile.default_invoice_template").alias("billing_default_invoice_template"),
        col("billing_profile.default_credit_memo_template").alias("billing_default_credit_memo_template"),
        col("billing_profile.entity_use_code").alias("billing_entity_use_code"),
        col("billing_profile.qbo_bill_parent_customer").cast("boolean").alias("billing_qbo_bill_parent_customer"),
        col("billing_profile.tax_exemption_reason").alias("billing_tax_exemption_reason")
    )

def normalize_transactions_df(df):
    """
    Normalize/casts for transactions resource based on provided definition.
    Many numeric-like fields come as strings in the API; cast to double where appropriate.
    Flatten auditentry if present and sanitize complex fields later.
    """
    return df.select(
        col("id"),
        col("number"),
        to_date(col("start_date"), "yyyy-MM-dd").alias("start_date"),
        to_date(col("end_date"), "yyyy-MM-dd").alias("end_date"),
        col("duration"),
        col("foreign_exchange_rate").cast("double").alias("foreign_exchange_rate"),
        col("home_amount").cast("double").alias("home_amount"),
        col("local_amount").cast("double").alias("local_amount"),
        col("home_normalized_amount").cast("double").alias("home_normalized_amount"),
        col("local_normalized_amount").cast("double").alias("local_normalized_amount"),
        col("home_arr_amount").cast("double").alias("home_arr_amount"),
        col("local_arr_amount").cast("double").alias("local_arr_amount"),
        col("always_use_amount_as_normalized_amount").cast("boolean").alias("always_use_amount_as_normalized_amount"),
        col("quantity").cast("double").alias("quantity"),
        col("term_number").cast("long").alias("term_number"),
        col("unbalanced_revenue_exception").cast("boolean").alias("unbalanced_revenue_exception"),
        col("different_invoice_item_permitted").cast("boolean").alias("different_invoice_item_permitted"),
        col("renewal_probability"),
        col("renewal_amount_value").cast("double").alias("renewal_amount_value"),
        col("renewal_amount_percentage").cast("double").alias("renewal_amount_percentage"),
        col("renewal_amount").cast("double").alias("renewal_amount"),
        col("order_number"),
        to_date(col("order_date"), "yyyy-MM-dd").alias("order_date"),
        col("recognize").cast("boolean").alias("recognize"),
        col("flagged"),
        col("notes"),
        col("invoice_description"),
        col("do_not_sync_invoices").cast("boolean").alias("do_not_sync_invoices"),
        col("home_rate").cast("double").alias("home_rate"),
        col("local_rate").cast("double").alias("local_rate"),
        col("home_normalized_rate").cast("double").alias("home_normalized_rate"),
        col("local_normalized_rate").cast("double").alias("local_normalized_rate"),
        col("renewal_quantity").cast("double").alias("renewal_quantity"),
        col("renewal_duration"),
        col("is_autorenewal").cast("boolean").alias("is_autorenewal"),
        col("reconcile_required").cast("boolean").alias("reconcile_required"),
        col("revenue_notes"),
        col("revenue_amount").cast("double").alias("revenue_amount"),
        to_timestamp(col("modified")).alias("modified"),
        col("sf_group_id"),
        col("sf_id"),
        col("sf_opportunity_id"),
        col("sf_opportunity_line_item_id"),
        col("sf_renewal_opportunity_id"),
        col("sf_renewal_opportunity_line_item_id"),
        col("cancelled").cast("boolean").alias("cancelled"),
        col("contract").cast("long").alias("contract"),
        col("item").cast("long").alias("item"),
        col("billing_method").cast("long").alias("billing_method"),
        col("ili_qb_class").cast("long").alias("ili_qb_class"),
        col("renew_using_item").cast("long").alias("renew_using_item"),
        col("renewal_factor").cast("double").alias("renewal_factor"),
        col("autorenewal_profile").cast("long").alias("autorenewal_profile"),
        col("sf_renewal_opportunity_rule").cast("long").alias("sf_renewal_opportunity_rule"),
        col("item_class").cast("long").alias("item_class"),
        col("project"),
        col("renewal_of_set"),
        col("is_renewed"),
        col("renewal_billing_method").cast("long").alias("renewal_billing_method"),
        col("crm_opportunity_id"),
        col("crm_opportunity_line_item_id"),
        col("hubspot_renewal_deal_rule").cast("long").alias("hubspot_renewal_deal_rule"),
        col("advanced_billing_subscription_id"),
        col("advanced_billing_subscription_line_id"),
        col("text_field1"),
        col("text_field2"),
        col("number_field1"),
        col("number_field2"),
        col("sales_rep"),
        col("sales_manager"),
        col("conversion"),
        # Flatten auditentry if present
        col("auditentry.created").alias("audit_created"),
        col("auditentry.created_by").alias("audit_created_by"),
        col("auditentry.modified").alias("audit_modified"),
        col("auditentry.modified_by").alias("audit_modified_by")
    )

def normalize_revenue_entries_df(df):
    """
    Normalize revenue_entries resource:
    {
      "id": 0,
      "transaction": 0,
      "start_date": "2019-08-24",
      "end_date": "2019-08-24",
      "home_amount": "string",
      "local_amount": "string",
      "modified": "2019-08-24T14:15:22Z"
    }
    """
    return df.select(
        col("id"),
        col("transaction").cast("long").alias("transaction"),
        to_date(col("start_date"), "yyyy-MM-dd").alias("start_date"),
        to_date(col("end_date"), "yyyy-MM-dd").alias("end_date"),
        col("home_amount").cast("double").alias("home_amount"),
        col("local_amount").cast("double").alias("local_amount"),
        to_timestamp(col("modified")).alias("modified")
    )

def write_invoice_lines_from_df(spark, df_invoices, out_line_items_path):
    """
    Explode 'line_items' from invoices dataframe and write normalized invoice lines to parquet.
    """
    line_items_df = df_invoices.select(
        col("id").alias("invoice_id"),
        explode_outer(col("line_items")).alias("line_item")
    ).select(
        col("invoice_id"),
        col("line_item.id").alias("line_id"),
        col("line_item.number").alias("line_number"),
        col("line_item.home_amount").cast("double").alias("home_amount"),
        col("line_item.local_amount").cast("double").alias("local_amount"),
        col("line_item.quantity").cast("double").alias("quantity"),
        col("line_item.notes").alias("notes"),
        col("line_item.no_transaction_permitted").cast("boolean").alias("no_transaction_permitted"),
        to_timestamp(col("line_item.exported_date")).alias("exported_date"),
        col("line_item.qb_txn_line_id").alias("qb_txn_line_id"),
        to_timestamp(col("line_item.sync_date")).alias("sync_date"),
        col("line_item.deleted_in_qb").cast("boolean").alias("deleted_in_qb"),
        to_timestamp(col("line_item.qb_time_modified")).alias("qb_time_modified"),
        col("line_item.sf_id").alias("sf_id"),
        col("line_item.item").cast("long").alias("item"),
        col("line_item.transaction").cast("long").alias("transaction"),
        col("line_item.qb_class").cast("long").alias("qb_class"),
        col("line_item.external_id").alias("external_id"),
        to_timestamp(col("line_item.modified")).alias("modified"),
        col("line_item.modified_by_name").alias("modified_by_name")
    )

    print(f"Ecriture des line_items vers {out_line_items_path} ...")
    line_items_df.write.mode("overwrite").parquet(out_line_items_path)
    print("Invoice lines écrits.")

def write_revenue_entries_for_transactions(spark, sc, transactions_raw_df, out_line_items_path, transactions_endpoint, headers):
    """
    For each transaction id in transactions_raw_df, call:
      GET {transactions_endpoint.rstrip('/')}/{transaction_id}/revenue_entries/
    Collect all revenue entries across transactions, normalize and write as parquet.
    """
    import requests
    from requests.adapters import HTTPAdapter
    from urllib3.util.retry import Retry

    # extract transaction ids (distinct)
    ids_rdd = transactions_raw_df.select("id").rdd.map(lambda r: r[0]).distinct()

    # quick check: if no ids, skip
    try:
        if ids_rdd.isEmpty():
            print("Aucun transaction id pour récupérer des revenue_entries. Skip.")
            return
    except Exception:
        # fallback: try take(1)
        if len(ids_rdd.take(1)) == 0:
            print("Aucun transaction id pour récupérer des revenue_entries. Skip.")
            return

    b_headers = sc.broadcast(headers)
    base_transactions = transactions_endpoint.rstrip('/')

    def partition_fetch_revenue(it):
        import requests
        import time
        from requests.adapters import HTTPAdapter
        from urllib3.util.retry import Retry

        session = requests.Session()
        retries = Retry(total=5, backoff_factor=0.5, status_forcelist=[429,500,502,503,504], raise_on_status=False)
        session.mount("https://", HTTPAdapter(max_retries=retries))
        session.mount("http://", HTTPAdapter(max_retries=retries))

        headers_loc = b_headers.value
        for iid in it:
            # build revenue entries url for this transaction
            rev_url = f"{base_transactions}/{iid}/revenue_entries/"
            try:
                items = list_resources_via_results(session, rev_url, headers_loc, page_size=PAGE_SIZE)
            except Exception as e:
                # on error, skip this transaction but log
                print(f"[WARN] erreur liste revenue_entries pour transaction {iid}: {e}")
                items = []
            for entry in items:
                # ensure the entry contains back ref to transaction if not present
                if isinstance(entry, dict) and "transaction" not in entry:
                    entry["transaction"] = iid
                yield entry

    rev_rdd = ids_rdd.mapPartitions(partition_fetch_revenue)
    # check if any revenue entries found
    sample = rev_rdd.take(1)
    if len(sample) == 0:
        print("Aucun revenue_entry trouvé pour les transactions. Skip writing revenue_entries parquet.")
        return

    json_rdd = rev_rdd.map(lambda d: json.dumps(d, ensure_ascii=False))
    rev_df = spark.read.json(json_rdd)

    print("Inferred schema for revenue_entries:")
    rev_df.printSchema()
    print(f"Count (approx) revenue_entries: {rev_df.count()}")

    # normalize and sanitize
    rev_df_norm = normalize_revenue_entries_df(rev_df)
    rev_df_norm = _sanitize_complex_columns(rev_df_norm)

    print(f"Ecriture des revenue_entries vers {out_line_items_path} ...")
    rev_df_norm.write.mode("overwrite").parquet(out_line_items_path)
    print("Revenue entries écrits.")

def main():
    spark = SparkSession.builder.appName("saasoptics-resources-extract").getOrCreate()
    sc = spark.sparkContext

    headers = build_headers()

    # Process customers
    process_resource(
        spark=spark,
        sc=sc,
        name="customers",
        endpoint_url=CUSTOMERS_ENDPOINT,
        headers=headers,
        normalize_func=normalize_customers_df
    )

    # Process items
    process_resource(
        spark=spark,
        sc=sc,
        name="items",
        endpoint_url=ITEMS_ENDPOINT,
        headers=headers,
        normalize_func=normalize_items_df
    )

    # Process invoices (write invoice lines from raw_df and write flat invoices parquet)
    process_resource(
        spark=spark,
        sc=sc,
        name="invoices",
        endpoint_url=INVOICES_ENDPOINT,
        headers=headers,
        normalize_func=normalize_invoices_df
    )

    # Process transactions (write revenue entries for each transaction and write flat transactions parquet)
    process_resource(
        spark=spark,
        sc=sc,
        name="transactions",
        endpoint_url=TRANSACTIONS_ENDPOINT,
        headers=headers,
        normalize_func=normalize_transactions_df
    )

    # Process contracts
    process_resource(
        spark=spark,
        sc=sc,
        name="contracts",
        endpoint_url=CONTRACTS_ENDPOINT,
        headers=headers,
        normalize_func=normalize_contracts_df
    )

    print("All resources processed.")
    # spark.stop()  # keep Spark session alive if running interactively

if __name__ == "__main__":
    main()